# Universidad Central · Big Data · Sesión 13
# ¿Cuándo basta pandas y cuándo necesitamos distribuir datos y modelos?

**Caso continuo:** Laura / Compras Claras. **Entorno principal:** Codespaces — elegir perfil **S13 · Dask, MapReduce y modelos SECOP**.

[Presentación y tutorial S13](../Presentaciones/s13-de-una-maquina-a-un-cluster.html#s1) · [Datos de referencia](../Datos/tc2_corpus_2026.json)

**No dependemos de que S12 ya se haya impartido.** El Taller 2 es una actividad evaluativa con su propia fecha. Aquí utilizamos dos fuentes versionadas:
- **200 expedientes** del Taller 2 para continuidad narrativa, *sin etiqueta de modalidad*.
- **1.000 registros SECOP** para buscar fichas y clasificar segmentos UNSPSC, *no de fraude*.

Al final obtendrás datos Parquet, un manifiesto de origen, una comparación real de workers, una representación TF-IDF y una explicación sobre cuándo Dask agrega valor. Todos los registros base son del repositorio; no necesitas APIs ni cuentas de pago.


## Objetivo de la sesión: demostrar qué reparte Dask y cuándo conviene

Un mismo caso SECOP articula tres preguntas: ¿cómo divido los textos en lotes?, ¿cómo el scheduler reparte el trabajo entre procesos worker?, ¿cómo convierto los textos a características TF-IDF para buscarlos o clasificarlos?

**Ruta principal:** inspeccionar CPU y memoria → ejecutar ocho lotes secuenciales y con Dask real → variar workers → entender particiones/MapReduce → calcular TF-IDF manual y con scikit-learn → guardar Parquet y evidencia JSON → justificar arquitectura.

**Evidencia propia:** direcciones de workers, comparación entre los dos modos de procesar lotes, salida idéntica y tabla TF-IDF. La auditoría de calidad de la fuente, los modelos neuronales y los benchmarks de 50.000 pertenecen a la preparación del docente; no se ejecutan en esta clase.

Las demostraciones usan datos versionados; no es necesario contar con un servicio OCR ni instalar Ollama.


## Mapa compacto del laboratorio

1. **Preparación:** Codespace S13, CPU lógicas, RAM y verificación de un worker verdadero.
2. **Concurrencia y trabajo real:** ocho lotes; modo espera **simulada** y modo texto **real** con los mismos datos SECOP. Se compara tiempo de arranque, ejecución y total.
3. **Particiones y MapReduce:** mismo resultado con pandas y Dask; promedio ponderado sin el error de promediar promedios.
4. **Representación de texto:** TF, DF, IDF, normalización L2, búsqueda coseno y clasificación UNSPSC.
5. **Salida:** Parquet, evidencia de los workers y decisión estructurada.

[Presentación y herramientas interactivas S13](../Presentaciones/s13-de-una-maquina-a-un-cluster.html#s1)

Para abrir Streamlit desde la raíz del repositorio:

```bash
streamlit run Laboratorios/S13_Codespaces/app_tfidf_unspsc.py --server.address 0.0.0.0 --server.port 8503
```

Abre **PORTS → 8503 → Private**. Los cronómetros históricos de 50.000 NO se vuelven a ejecutar.


## 0 · Decidir antes de instalar

**Pregunta profesional:** ¿qué problema justifica pagar el costo de coordinar trabajadores?

| Trabajo | Primera opción razonable |
|---|---|
| 200 registros que caben en memoria | pandas |
| Consulta SQL sobre muchos CSV/Parquet | considerar DuckDB |
| Muchas transformaciones independientes o datos mayores que memoria | Dask puede ayudar |
| Un modelo clásico con cientos de ejemplos | scikit-learn |
| Un único modelo profundo entrenado entre máquinas | algoritmo con soporte distribuido explícito |

**OJO:** Dask no es sinónimo de «más rápido». Dos workers dentro de un Codespace **no son dos computadores**. Cada herramienta resuelve una necesidad distinta.


## 1 · Crear tu entorno propio en el navegador

**HAZ ESTO AHORA.** 1. Abre [el repositorio](https://github.com/jazaineam1/BigData2026). 2. Pulsa **Code → Codespaces → ⋯ → New with options**. 3. En configuración del contenedor selecciona **S13 · Dask, MapReduce y modelos SECOP**. **No** elijas «Airflow Lab» (predeterminado) ni «S11 multimedia». 4. Crea un Codespace y abre una **Terminal → New Terminal**.

Dentro de la terminal ejecuta:

```bash
python --version
python -m pip show pandas
python -m pip show dask
python -m pip show scikit-learn
python Laboratorios/S13_Codespaces/verificar_entorno.py
```

Si falta una biblioteca, **dentro de Codespaces**, no en tu PC: `python -m pip install -r Laboratorios/S13_Codespaces/requirements.txt`. Abre este cuaderno en VS Code web y selecciona el kernel Python del Codespace. No necesita Ubuntu, Docker ni VS Code instalados localmente.

**Resultados:** debe terminar con `LISTO PARA S13`; si escogiste el perfil equivocado, crea otro Codespace con la configuración S13.


### 1A · Preparar rutas (por qué no importa si abriste el cuaderno desde Cuadernos)

`Path` maneja rutas; `sys.path` indica dónde encontrar las funciones reutilizables. Conserva visible el código: puedes adaptarlo a un proyecto ajeno.


In [ ]:
from pathlib import Path
import sys

RAIZ = Path.cwd()
if not (RAIZ / "Datos").exists():
    RAIZ = RAIZ.parent

assert (RAIZ / "Datos/tc2_corpus_2026.json").exists(), "Abre el repositorio en Codespaces"
sys.path.insert(0, str(RAIZ / "Laboratorios/S13_Codespaces"))
print("Raíz del curso:", RAIZ)


### 1B · Comprobar biblioteca y archivos, no sólo que pip haya respondido

Esta celda ejecuta un cálculo mínimo Dask y comprueba que existen las dos fuentes del curso.


In [ ]:
import pandas as pd
import dask.dataframe as dd
from analisis import leer_corpus, leer_etiquetados

demo = pd.DataFrame({"grupo": ["A", "A", "B"], "valor": [10, 20, 30]})
mini = dd.from_pandas(demo, npartitions=2)
print(mini.groupby("grupo")["valor"].sum().compute())
assert int(mini.groupby("grupo")["valor"].sum().compute().loc["A"]) == 30
print("Fuentes y herramientas:", len(leer_corpus()), len(leer_etiquetados()))


**Cómo se lee:** A suma 30; B suma 30. Se encuentran 200 expedientes de referencia y 1.000 procesos etiquetados.  
**Qué nos dice:** pandas y Dask están disponibles y pueden ejecutar un cálculo.  
**Qué NO permite concluir:** que haya un clúster físico o mayor rendimiento que pandas.  
**Error común:** asumir que «pip instaló» equivale a «el flujo completo funciona».


## Laboratorio central · primero un caso en que Dask puede demostrar concurrencia

**Experimento inicial del estudiante:** trabajamos con ocho lotes independientes tomados de hasta 1.000 títulos SECOP. La función calcula un conteo real y huella SHA-256; incluye una espera de 0,6 segundos **simulada y explícita**, análoga a esperar respuesta de un servicio de OCR/ASR. Dask Distributed **sí ejecuta** la función en varios procesos worker; no ejecutamos OCR ni transcripción en esta demostración.

**Qué diferencia debes observar:** el tiempo secuencial ejecuta los ocho lotes uno por uno; Dask caliente distribuye entre workers; el total añade el arranque del scheduler. El script comprueba que ambos caminos generen el mismo resultado por lote. La ganancia puede variar según VM, número de workers y carga del Codespace.

**Códigos útiles del entorno:**

```bash
nproc
lscpu | head -25
free -h
python Laboratorios/S13_Codespaces/dask_cuando_si.py --workers 2
```

Si `nproc` indica al menos cuatro CPU lógicas y la cuota permite la prueba, compara con `--workers 4`. Este laboratorio no descarga modelos ni procesa 50.000 fichas.


In [ ]:
import os
import subprocess
import sys
import json
from pathlib import Path

print("CPU lógicas del sistema:", os.cpu_count())
disponibles = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
print("CPU lógicas permitidas por afinidad:", disponibles)
from dask_cuando_si import cpus_disponibles
print("Límite operativo orientativo (afinidad + cuota cgroup):", cpus_disponibles())
print("Para inspeccionar físicamente la VM ejecuta en terminal: nproc; lscpu; free -h")
print("Una VM puede tener 1, 2 o 4 workers, cada uno con uno o varios hilos.")


In [ ]:
# Demo pedagógica pequeña: 8 tareas con Dask Distributed real y espera externa SIMULADA.
# No activa Ollama, 50.000 fichas ni ningún servicio externo.
DEMO_DASK_EN_RUN_ALL = True
WORKERS_DEMO_S13 = 2  # Cambia SOLO este número a 4 para repetir la demostración.
script_dask = RAIZ / "Laboratorios/S13_Codespaces/dask_cuando_si.py"
assert script_dask.exists(), script_dask
if DEMO_DASK_EN_RUN_ALL:
    prueba = subprocess.run(
        [sys.executable, str(script_dask), "--workers", str(WORKERS_DEMO_S13),
         "--tareas", "8", "--espera", "0.6"],
        check=True, cwd=RAIZ, capture_output=True, text=True, timeout=120,
    )
    print(prueba.stdout)
    marca = "RESULTADO_S13_DASK="
    linea = next((l[len(marca):] for l in prueba.stdout.splitlines()
                  if l.startswith(marca)), None)
    assert linea, "No se obtuvo JSON verificable de Dask"
    resumen_demo = json.loads(linea)
    assert resumen_demo["workers"] == WORKERS_DEMO_S13
    assert resumen_demo["resultados_identicos"]
    assert len(resumen_demo["workers_reales"]) == WORKERS_DEMO_S13
    print("Ganador con clúster caliente:", resumen_demo["ganador_caliente"])
    print("Ganador incluyendo arranque:", resumen_demo["ganador_total"])
else:
    print("Para activar esta demostración ligera cambia DEMO_DASK_EN_RUN_ALL=True.")


In [ ]:
# Procesamiento REAL de 1000 títulos SECOP: conteo de vocabulario por lote.
# Se usan los mismos trabajadores; no hay espera simulada, ni OCR, ni modelos.
prueba_texto = subprocess.run(
    [sys.executable, str(script_dask), "--workers", str(WORKERS_DEMO_S13),
     "--tareas", "8", "--modo", "texto"], check=True, cwd=RAIZ,
    capture_output=True, text=True, timeout=120,
)
linea = next((l.split("RESULTADO_S13_DASK=", 1)[1]
              for l in prueba_texto.stdout.splitlines()
              if l.startswith("RESULTADO_S13_DASK=")), None)
assert linea, prueba_texto.stdout[-1800:]
resumen_texto = json.loads(linea)
assert resumen_texto["resultados_identicos"]
assert sum(resumen_texto["terminos_distintos_por_lote"]) > 0
print("Títulos SECOP:", resumen_texto["filas_procesadas"])
print("Workers reales:", len(resumen_texto["workers_reales"]))
print("Segundos secuencial:", resumen_texto["segundos_secuencial"])
print("Segundos Dask caliente:", resumen_texto["segundos_dask_cluster_caliente"])
print("Segundos Dask con arranque:", resumen_texto["segundos_dask_total_incluido_arranque"])
print("Ganador con arranque:", resumen_texto["ganador_total"])


**Cómo se lee:** esta vez Dask hace una transformación auténtica: cuenta los términos de hasta 1.000 títulos SECOP, reparte lotes entre workers y verifica igualdad de diccionarios de frecuencias. No hay pausas artificiales.

**Qué nos dice:** una tarea pequeña puede perder frente a la ejecución local por el costo de iniciar y coordinar workers.

**Qué NO permite concluir:** que Dask no sirva cuando hay datos mayores, operaciones costosas o múltiples máquinas.

**Error común:** confundir ganar en una espera concurrente simulada con ganar en cualquier función de procesamiento.


**Cómo se lee:** el JSON indica número real de workers, CPU habilitadas, filas y palabras procesadas, tiempo secuencial, tiempo de arranque, tiempo Dask caliente y tiempo Dask completo.  
**Qué demuestra:** Dask ejecutó trabajo real en procesos worker y el resultado fue idéntico. La espera fue **simulada**, no es una medición real de OCR.  
**Qué no demuestra:** que Dask sea siempre más rápido en cualquier equipo o con un modelo neuronal.  
**Prueba adicional:** modifica `--workers 2` a `--workers 4` solo si la VM habilita al menos 4 CPU lógicas; observa si cambia la aceleración y el consumo de memoria.

**¿Por qué enseñamos Dask?** Porque puede distribuir funciones Python repetibles, procesar particiones de datos y coordinar tareas independientes. A partir de ahora podrás sustituir la función de demostración por una función real de extracción o clasificación por archivo.


## 2 · Sistemas de archivos: bloques, nodos y réplicas

Un sistema como **HDFS** divide archivos en bloques lógicos y almacena esos bloques en nodos. La replicación crea copias para tolerar fallos. El **NameNode** administra metadatos de ubicación; los **DataNodes** guardan bloques.

Esto NO es lo mismo que partir un DataFrame para procesarlo. La siguiente celda es una **SIMULACIÓN en Python**, no un clúster Hadoop real.

| Elemento | Función |
|---|---|
| Nodo | máquina o proceso que participa |
| Bloque | unidad de almacenamiento |
| Réplica | otra copia de un mismo bloque |
| Partición | subconjunto lógico de trabajo |
| Scheduler | coordinador de tareas |
| Worker | ejecutor de tareas |

**Decisión:** ¿qué sucede si se pierde un nodo que guarda la única copia de un bloque?


In [ ]:
ubicaciones = {
    "bloque_1": ["nodo_A", "nodo_B"],
    "bloque_2": ["nodo_B", "nodo_C"],
    "bloque_3": ["nodo_A", "nodo_C"],
}
nodo_caido = "nodo_B"

for bloque, nodos in ubicaciones.items():
    disponibles = [n for n in nodos if n != nodo_caido]
    print(bloque, "→", disponibles)
    assert disponibles, f"Bloque no recuperable: {bloque}"


**Cómo se lee:** cada bloque conserva una ubicación distinta de nodo_B.  
**Qué nos dice:** copiar bloques en distintos nodos puede proteger contra una falla simple.  
**Qué NO permite concluir:** que un Codespace sea un HDFS real ni que todas las políticas de replicación garanticen disponibilidad.  
**Error común:** confundir replicar los datos con paralelizar el cálculo.


## 3 · MapReduce: la trampa del promedio de promedios

Imagina 3 lotes de contrataciones con diferentes tamaños. En el primer lote hay 2 importes, en el segundo 5 y en el tercero 100. **Promediar las tres medias de los lotes es incorrecto** porque no representan el mismo número de observaciones.

`map` produce estadísticas parciales. `reduce` combina parciales sin perder los pesos. **Es el modelo MapReduce implementado en Python**, no un servicio de Apache Hadoop.


In [ ]:
lotes = [
    [100, 120],
    [10, 12, 11, 9, 8],
    [1] * 100,
]
parciales = [(sum(lote), len(lote)) for lote in lotes]
print("MAP (suma, cantidad):", parciales)


In [ ]:
suma_total = sum(suma for suma, cantidad in parciales)
cantidad_total = sum(cantidad for suma, cantidad in parciales)

promedio_correcto = suma_total / cantidad_total
promedio_incorrecto = sum(s / n for s, n in parciales) / len(parciales)

print("REDUCE correcto:", round(promedio_correcto, 3))
print("Promedio de promedios:", round(promedio_incorrecto, 3))
assert round(promedio_correcto, 6) == round(370/107, 6)


**Cómo se lee:** 370 unidades en 107 observaciones dan una media de aproximadamente 3,458.  
**Qué nos dice:** para un promedio distribuido necesitamos **suma y cantidad**, no sólo medias locales.  
**Qué NO permite concluir:** que ya usamos Hadoop MapReduce ni que un algoritmo cualquiera sea divisible así.  
**Error común:** `mean(mean(lote))` ignora los tamaños de las particiones.

**Transfiere:** esta estrategia sirve para métricas de ventas, llamadas o tiempos por cliente, siempre que conserves los pesos.


## 4 · Dataset preparado para demostrar particiones (sin auditoría estudiantil)

En S13 utilizaremos la **muestra SECOP versionada en el repositorio** como insumo para operaciones sobre tablas y textos. El material está preparado para el laboratorio: sus reglas de auditoría de fuente, ceros, identificadores y segmentos UNSPSC son asuntos de validación técnica docente, no actividades evaluables para el estudiante.

En clase importa **cómo Dask procesa una función sobre bloques, cómo se distribuye en varios workers, cómo se combina el resultado y cuánto tiempo cuesta hacerlo**.


In [ ]:
referencia = leer_corpus()
print("Expedientes del taller:", len(referencia))
print(referencia[["id_proceso", "entidad"]].head(3))
assert referencia["id_proceso"].is_unique


### 4A · Datos para el problema supervisado

El CSV de SECOP versionado sí tiene `descripci_n_del_procedimiento` y `modalidad_de_contratacion`.

**Pregunta del modelo:** ¿podemos aprender a clasificar una modalidad registrada a partir del texto? **No es un detector de corrupción**.


In [ ]:
tabla = leer_etiquetados()
print("Dimensiones:", tabla.shape)
print(tabla["modalidad_de_contratacion"].value_counts().head(7))
print(tabla[["id_del_proceso", "descripci_n_del_procedimiento"]].head(2))
assert tabla["id_del_proceso"].is_unique


**Cómo se lee:** 1.000 filas, 59 columnas; «Contratación directa» y «Contratación régimen especial» son clases frecuentes.  
**Qué nos dice:** hay etiquetas registradas y texto para plantear una clasificación didáctica.  
**Qué NO permite concluir:** que esas frecuencias representen todo SECOP ni que el texto contenga evidencia de fraude.  
**Error común:** entrenar sobre los 200 expedientes del Taller 2 como si tuvieran esa etiqueta.


## 5 · pandas primero: la referencia de corrección (conteo de modalidades, NO clasificación)

`groupby` agrupa filas por modalidad y `size()` cuenta filas. Esta salida es la referencia contra la cual compararemos Dask.


In [ ]:
conteo_pandas = tabla.groupby("modalidad_de_contratacion").size().sort_index()
print(conteo_pandas.head(10))
print("Total:", int(conteo_pandas.sum()))
assert conteo_pandas.sum() == 1000


## 6 · Dask DataFrame: el mismo cálculo por particiones

`from_pandas` divide el DataFrame en particiones. `groupby` construye operaciones diferidas; `compute()` materializa el resultado. Una partición **no** es un servidor.

¿Cuál sería el costo de Dask en sólo mil registros? El coordinador, el grafo de tareas y la comunicación pueden tardar más que la operación real.


In [ ]:
dask_tabla = dd.from_pandas(tabla, npartitions=4)
pendiente = dask_tabla.groupby("modalidad_de_contratacion").size()

print("Particiones:", dask_tabla.npartitions)
print("Antes de compute:", type(pendiente).__name__)
conteo_dask = pendiente.compute().sort_index()
valores_pandas = {str(k): int(v) for k, v in conteo_pandas.items()}
valores_dask = {str(k): int(v) for k, v in conteo_dask.items()}
assert valores_pandas == valores_dask
print("Resultado Dask:", conteo_dask.head(), "\nCoincide: SÍ")


### 6A · Medir sin inventar «Dask siempre gana»

Una medición debe incluir el tiempo de ejecución, el número de particiones y el tamaño del conjunto. Se repite en condiciones comparables; el primer tiempo puede incluir inicialización, cachés y procesos. Comparamos por **corrección** antes de velocidad.


In [ ]:
from analisis import comparar_pandas_dask
mediciones = []
for n_particiones in (1, 2, 4):
    mediciones.append(comparar_pandas_dask(tabla, npartitions=n_particiones))

display(pd.DataFrame(mediciones)[
    ["filas", "particiones", "segundos_pandas", "segundos_dask", "coinciden"]
])


**Cómo se lee:** los segundos se miden en TU Codespace y pueden cambiar en cada ejecución.  
**Qué nos dice:** la comparación es local, sobre el mismo conjunto de 1.000 registros.  
**Qué NO permite concluir:** cuál herramienta es más rápida para millones de filas en otro sistema.  
**Error común:** confundir aumentar particiones con aumentar máquinas o comparar ejecuciones con cargas distintas.


## 10 · Guardar evidencia reutilizable con Parquet

Parquet es un formato **columnar**. El archivo siguiente mantiene procesos únicos, modalidad y descripción; no guarda por sí mismo los pesos de un modelo entrenado. La sesión 14 lo leerá con PySpark.


In [ ]:
from analisis import escribir_resultados
manifest = escribir_resultados(tabla)
print("Salida:", manifest["salida"])
print("Filas:", manifest["filas"])
print("SHA-256 fuente:", manifest["sha256_fuente"][:20], "...")

archivo = RAIZ / "Laboratorios/S13_Codespaces/salidas/secop_s13.parquet"
reabierto = pd.read_parquet(archivo)
assert reabierto["id_del_proceso"].is_unique
print("Reabierto:", reabierto.shape)


**Cómo se lee:** se exportaron 1.000 procesos únicos de la muestra y se recuperaron desde Parquet.  
**Qué nos dice:** la entrega es tabular, trazable a una fuente específica y legible.  
**Qué NO permite concluir:** que el archivo Parquet tenga los modelos entrenados o preserve una licencia externa no registrada.  
**Error común:** confundir igualdad lógica de filas con igualdad byte a byte de dos archivos Parquet reescritos.


## 11 · Transferencia: convierte la técnica en tu propio proyecto

No memorices rutas del curso. En un proyecto nuevo identifica:

- **Archivo de entrada**: CSV o Parquet que controles y tengas permiso de usar.
- **Clave única**: ID que representa una observación.
- **Columna de agrupación**: qué necesitas comparar y por qué.
- **Etiqueta**: una variable que **exista realmente** si vas a entrenar supervisadamente.
- **Texto o variables**: de dónde provienen; ¿alguna revela directamente la etiqueta?
- **Control simple**: pandas antes de Dask.
- **Salida**: Parquet + manifiesto con fuente, conteos y decisiones.

En `Laboratorios/S13_Codespaces/analisis.py` puedes reemplazar `CSV`, `COL_TEXTO`, `COL_LABEL` y `CLASES` después de inspeccionar el nuevo esquema. Si la columna no existe, debes recibir un error visible, no fabricar valores.


## 12 · Autocomprobación técnica

Elige una herramienta para cada tarea **después de observar los resultados**, no sólo por definición:

| Caso | Primera respuesta defendible |
|---|---|
| 200 filas para un conteo | pandas |
| SQL sobre archivos grandes | considerar DuckDB |
| Varias transformaciones y tareas independientes | Dask según volumen y medición |
| Un modelo clásico pequeño | scikit-learn |
| Muchos entrenamientos de parámetros independientes | Dask Futures si la carga justifica coordinación |
| Un único modelo aprendido entre nodos | estimador con soporte distribuido explícito |

**Para llevar:** *procesar datos por particiones* ≠ *ejecutar varios modelos* ≠ *entrenar un modelo único de forma distribuida*.


## Aplicación de TF-IDF al mismo conjunto SECOP

El objetivo supervisado será **únicamente el segmento UNSPSC declarado**, derivado del título y la descripción. La modalidad contractual solo se utilizó antes para ilustrar el `groupby` entre pandas y Dask; no se entrena un segundo clasificador. Esto evita introducir dos preguntas predictivas distintas.

La ruta principal consiste en **calcular TF-IDF manual**, contrastarlo con scikit-learn y realizar búsqueda por coseno. La clasificación UNSPSC queda como **ampliación opcional**, sin entrenamiento automático en Run All. Las mediciones grandes, XGBoost y EmbeddingGemma son profundizaciones exclusivas del docente.


## ¿Por qué TF-IDF pertenece a Big Data?

Un sistema de búsqueda o clasificación no puede recibir directamente frases en español como si fueran números. TF-IDF convierte miles de textos en una **matriz de características**: filas = documentos, columnas = términos, valores = importancia de cada término en cada documento. Dask puede repartir partes del procesamiento de esa matriz, pero no cambia la definición matemática de los pesos.

Para tres textos de ejemplo: A = «mantenimiento computadores», B = «infraestructura computadores», C = «hospital mantenimiento». DF de `hospital` es 1 e IDF suavizado 1,6931; sin embargo en A su TF es 0 y su peso final es 0. `mantenimiento` y `computadores` aparecen en dos textos, por lo que tienen IDF 1,2877; en A, sus pesos finales normalizados L2 son 0,7071 cada uno. Se usa **L2** para comparar orientaciones de vectores en la similitud coseno y no dejar que la longitud de una ficha determine por sí sola el ranking.

El propósito es **buscar documentos relacionados y representar texto para modelos**, no «medir fraude». Cuando los documentos crecen, Dask puede ayudar a transformar particiones; la definición de TF-IDF permanece igual.


## Herramienta TF-IDF y búsqueda léxica (ruta principal)

**Misión:** representar títulos y descripciones SECOP como vectores y comprender la búsqueda por coseno. Se utiliza el minicorpus A/B/C común con las diapositivas y Streamlit.

**Práctica principal:** calcular TF, DF, IDF y L2, comprobar scikit-learn y consultar similitud entre documentos. **Opcional:** clasificación UNSPSC con LinearSVC y gráficos F1, después de dominar Dask. Ninguna parte del laboratorio principal predice fraude, precios ni legalidad.


### ¿Por qué calculamos TF-IDF?

Un ordenador no puede ordenar frases usando directamente sus palabras: las convertimos en vectores numéricos. **TF** cuenta apariciones en cada documento, **DF** cuántos documentos contienen el término e **IDF** reduce el peso de términos comunes:

\[
\mathrm{IDF}(t)=\ln\left(\frac{1+N}{1+DF(t)}\right)+1
\]

Multiplicamos TF × IDF y después dividimos por la norma **L2** para comparar vectores con coseno.

**Único minicorpus usado en presentación, cuaderno y Streamlit:** A = «mantenimiento computadores»; B = «infraestructura computadores»; C = «hospital mantenimiento». Los términos «computadores» y «mantenimiento» tienen DF=2 e IDF≈1,28768. «Hospital» e «infraestructura» tienen DF=1 e IDF≈1,69315. En A, «hospital» tiene TF=0, por lo que su peso es 0 a pesar del IDF mayor.


In [ ]:
import math
from collections import Counter
import pandas as pd

mini_documentos = [
    "mantenimiento computadores",
    "infraestructura computadores",
    "hospital mantenimiento",
]
tokens = [doc.lower().split() for doc in mini_documentos]
vocabulario = sorted(set(sum(tokens, [])))
N = len(tokens)
filas = []
for termino in vocabulario:
    df = sum(termino in doc for doc in tokens)
    idf = math.log((1 + N) / (1 + df)) + 1
    for num, doc in enumerate(tokens):
        tf = doc.count(termino)
        filas.append({
            "documento": "ABC"[num], "termino": termino,
            "TF": tf, "DF": df, "IDF": round(idf, 5),
            "TF×IDF_sin_normalizar": round(tf * idf, 5),
        })
pesos_manuales = pd.DataFrame(filas)
display(pesos_manuales)
assert set(pesos_manuales["DF"]) == {1, 2}
assert abs(pesos_manuales.query("termino == 'hospital'")["IDF"].iloc[0]
           - (math.log(4/2)+1)) < 1e-4


**Cómo se lee:** cada palabra del vocabulario aparece por documento; TF=0 si la palabra no está, DF cuenta documentos y el IDF solo depende de cuántos documentos usan la palabra.

**Qué nos dice:** «hospital» tiene IDF mayor que «computadores», pero en A su peso es cero.

**Qué NO permite concluir:** que un IDF alto sea una probabilidad o una medida de verdad del documento.

**Error común:** confundir TF con DF o creer que L2 suma pesos hasta 1 (L2 normaliza la raíz de cuadrados).


### Estación 2 · Contrastar la fórmula manual con scikit-learn

`TfidfVectorizer` primero construye un **vocabulario**, luego calcula la frecuencia documental, IDF, TF ponderado y normaliza cada fila. Usaremos `min_df=1` sólo para el ejemplo de tres documentos. En el benchmark grande `min_df=2` es razonable porque hay suficientes documentos.

**Mini ficha de funciones:**

| Función | Parámetros y devolución | Error frecuente |
|---|---|---|
| `fit()` | Aprende vocabulario e IDF del corpus de entrenamiento | Aplicarla sobre el test |
| `transform()` | Devuelve matriz dispersa de `filas × términos` | Interpretar un cero como dato faltante |
| `fit_transform()` | Hace ambas operaciones en entrenamiento | Repetirla sobre el test, cambiando columnas |
| `get_feature_names_out()` | Nombres de términos en orden de columnas | Suponer que cada vector es una etiqueta |


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

toy = TfidfVectorizer(
    token_pattern=r"(?u)\b\w+\b",
    min_df=1,
    norm="l2", smooth_idf=True, sublinear_tf=False,
)
mini_X = toy.fit_transform(mini_documentos)
mini_tabla = pd.DataFrame(
    np.round(mini_X.toarray(), 5),
    index=["A", "B", "C"],
    columns=toy.get_feature_names_out(),
)
display(mini_tabla)
assert mini_X.shape == (3, 4)
assert np.allclose(np.linalg.norm(mini_X.toarray(), axis=1), 1.0)


**Cómo se lee:** cada fila es el vector de un texto; cada columna es una palabra. Los valores ya están normalizados, por eso no son iguales a los productos TF×IDF sin normalizar de la celda anterior.  
**Qué nos dice:** podemos reproducir la aritmética y comprobar matemáticamente lo que entrega la biblioteca.  
**Qué NO permite concluir todavía:** que el algoritmo entiende contexto semántico o calidad del contrato.  
**Error común:** creer que los pesos de TF-IDF son probabilidades de cada categoría.

**Transferencia:** prueba `norm=None`, compara la matriz y explica por qué cambió.


### Estación 3 · Similitud coseno: buscar un expediente sin entrenar un clasificador

TF-IDF permite **recuperar textos relacionados** incluso sin etiquetas. La consulta y los documentos deben representarse con el mismo vocabulario. La similitud coseno mide su alineación:

\[
\cos(\theta) = \frac{a\cdot b}{\|a\|\,\|b\|}
\]

Con vectores normalizados L2, equivale al producto punto. En un corpus de TF-IDF no negativo suele estar entre 0 y 1; un 0 puede significar palabras distintas, no temas necesariamente diferentes.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

consulta = "quiero mantenimiento de computadores"
consulta_vector = toy.transform([consulta])
similares = cosine_similarity(consulta_vector, mini_X).ravel()
tabla_similitud = pd.DataFrame({
    "documento": ["A", "B", "C"],
    "texto": mini_documentos,
    "similitud_coseno": np.round(similares, 4),
}).sort_values("similitud_coseno", ascending=False)
display(tabla_similitud)
assert len(similares) == 3


**Cómo se lee:** el documento con coseno más alto comparte términos relevantes con la consulta.  
**Qué nos dice:** TF-IDF habilita una búsqueda léxica reproducible sin llamadas a modelos grandes.  
**Qué NO permite concluir todavía:** que frases con sinónimos no presentes sean irrelevantes.  
**Error común:** construir otro `TfidfVectorizer` para la consulta; los índices de columnas ya no coincidirían.

**Reto:** reemplaza «computadores» por «PC» y observa si la similitud cae. Esa limitación motiva probar EmbeddingGemma 2.


## Ampliación opcional · Clasificación UNSPSC a partir del texto

La clase principal de S13 termina con Dask Distributed, MapReduce, TF-IDF y búsqueda. Las siguientes celdas muestran cómo convertir la misma representación en un clasificador supervisado **solo si decides activar la ampliación**. No es necesario para cumplir la práctica ni se ejecuta automáticamente.

Para activar: en la próxima celda cambia `EJECUTAR_AMPLIACION_S13 = False` por `True` y ejecuta **solo las celdas de ampliación** hasta los gráficos. Se trabaja exclusivamente con la muestra pequeña SECOP versionada, sin 50.000 registros, Ollama ni XGBoost.


In [ ]:
EJECUTAR_AMPLIACION_S13 = False  # Activa True solo para esta ampliación.
if globals().get("EJECUTAR_AMPLIACION_S13", False):
    import re
    from pathlib import Path

    archivo_secop = RAIZ / "Cuadernos/datos/secop_chunks/prueba_chunk_0000000.csv"
    columnas_secop = [
        "id_del_proceso", "nombre_del_procedimiento",
        "descripci_n_del_procedimiento", "codigo_principal_de_categoria",
        "fecha_de_publicacion_del",
    ]
    secop_pequeno = pd.read_csv(
        archivo_secop, usecols=columnas_secop,
        dtype=str, keep_default_na=False,
    ).drop_duplicates("id_del_proceso")
    patron = re.compile(r"^(?:V[0-9]+\.)?([0-9]{8})$")
    def segmento_unspsc(codigo):
        m = patron.fullmatch(str(codigo).strip())
        return m.group(1)[:2] if m else None

    secop_pequeno["segmento"] = secop_pequeno[
        "codigo_principal_de_categoria"].map(segmento_unspsc)
    secop_pequeno["texto"] = (
        secop_pequeno["nombre_del_procedimiento"] + " " +
        secop_pequeno["descripci_n_del_procedimiento"]
    ).str.strip()
    print("Procesos únicos:", len(secop_pequeno))
    print("Sin segmento válido:", secop_pequeno["segmento"].isna().sum())
    display(secop_pequeno[["id_del_proceso", "segmento", "texto"]].head(5))


**Cómo se lee:** cada fila es una ficha de proceso; `texto` será la entrada para TF-IDF y `segmento` una categoría administrativa de demostración.  
**Qué nos dice:** podemos procesar un mismo esquema por partes y unir resultados.  
**Qué NO permite concluir:** que este registro contenga un dictamen jurídico ni que sea una prueba de fraude.  
**Error común:** convertir en objetivo del laboratorio una auditoría de datos que no es parte de la enseñanza de Dask.

**Continuamos:** veremos cómo repartir transformaciones y entrenamientos; no se solicita al estudiante elaborar un informe de calidad de datos.


### Ampliación optativa · Entrenar TF-IDF + LinearSVC

Si activaste la ampliación, la etiqueta es **únicamente el segmento UNSPSC declarado**. El vectorizador aprende vocabulario con el entrenamiento y usa `transform()` sobre el test posterior. La F1 macro es una medida de clasificación, **no una métrica de Dask ni una prueba jurídica**.

Si no activaste el bloque, no necesitas ejecutar nada aquí; continúa con la decisión estructurada de cierre.


In [ ]:
if globals().get("EJECUTAR_AMPLIACION_S13", False):
    from sklearn.svm import LinearSVC
    from sklearn.dummy import DummyClassifier
    from sklearn.metrics import f1_score
    from sklearn.pipeline import make_pipeline

    clases_objetivo = {"80", "85", "93", "81", "72"}
    ejemplos = secop_pequeno[
        secop_pequeno["segmento"].notna()
        & secop_pequeno["texto"].str.len().ge(20)
    ].copy()
    ejemplos["clase"] = ejemplos["segmento"].where(
        ejemplos["segmento"].isin(clases_objetivo), "OTRAS")
    ejemplos["fecha"] = pd.to_datetime(
        ejemplos["fecha_de_publicacion_del"], errors="coerce")
    ejemplos = ejemplos.dropna(subset=["fecha"]).sort_values(
        ["fecha", "id_del_proceso"])
    i = int(.8 * len(ejemplos))
    train_secop, test_secop = ejemplos.iloc[:i], ejemplos.iloc[i:]
    modelito = make_pipeline(
        TfidfVectorizer(max_features=3000, min_df=2, ngram_range=(1,2)),
        LinearSVC(class_weight="balanced", random_state=42),
    )
    modelito.fit(train_secop["texto"], train_secop["clase"])
    predicciones = modelito.predict(test_secop["texto"])
    control = DummyClassifier(strategy="most_frequent")
    control.fit(np.zeros((len(train_secop),1)), train_secop["clase"])
    base_pred = control.predict(np.zeros((len(test_secop),1)))
    etiquetas_fijas = ["72","80","81","85","93","OTRAS"]
    print("Train:",len(train_secop),"Test:",len(test_secop))
    print("F1 macro modelo:", round(f1_score(
        test_secop["clase"], predicciones,
        labels=etiquetas_fijas, average="macro", zero_division=0),4))
    print("F1 macro mayoritaria:", round(f1_score(
        test_secop["clase"], base_pred,
        labels=etiquetas_fijas, average="macro", zero_division=0),4))


### Ampliación optativa · Dos gráficos y comparación de n-gramas

La celda siguiente genera dos figuras y CSV de clasificación solo cuando la ampliación está activada: F1 macro con unigramas frente a bigramas y F1 por segmento. El conjunto de prueba es el mismo en ambas comparaciones; estos gráficos **no forman parte de la evidencia obligatoria de Dask**.


In [ ]:
if globals().get("EJECUTAR_AMPLIACION_S13", False):
    # Dos gráficos y tabla trazable, sobre SOLO la muestra pequeña (sin jobs pesados).
    import hashlib
    import matplotlib.pyplot as plt
    from sklearn.metrics import classification_report
    from sklearn.pipeline import make_pipeline
    from sklearn.svm import LinearSVC
    from sklearn.feature_extraction.text import TfidfVectorizer

    nombres = ["Unigramas", "Unigramas + bigramas"]
    configs = [(1, 1), (1, 2)]
    resultados_ngramas = []
    predicciones_por_config = {}
    for nombre, ngramas in zip(nombres, configs):
        pipeline = make_pipeline(
            TfidfVectorizer(max_features=3000, min_df=2, ngram_range=ngramas),
            LinearSVC(class_weight="balanced", random_state=42),
        )
        pipeline.fit(train_secop["texto"], train_secop["clase"])
        pred = pipeline.predict(test_secop["texto"])
        met = classification_report(
            test_secop["clase"], pred, labels=etiquetas_fijas,
            zero_division=0, output_dict=True,
        )
        predicciones_por_config[nombre] = met
        resultados_ngramas.append({
            "configuracion": nombre,
            "ngramas": str(ngramas),
            "f1_macro": float(met["macro avg"]["f1-score"]),
            "n_train": int(len(train_secop)),
            "n_test": int(len(test_secop)),
        })

    tabla_ngramas = pd.DataFrame(resultados_ngramas)
    test_hash = hashlib.sha256(
        "\n".join(test_secop["id_del_proceso"].astype(str)).encode("utf-8")
    ).hexdigest()
    tabla_ngramas["test_sha256_ids"] = test_hash
    carpeta_evidencia = RAIZ / "Laboratorios/S13_Codespaces/salidas"
    carpeta_evidencia.mkdir(parents=True, exist_ok=True)
    tabla_ngramas.to_csv(carpeta_evidencia / "s13_ngramas_mismo_test.csv", index=False)
    display(tabla_ngramas)

    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.bar(tabla_ngramas["configuracion"], tabla_ngramas["f1_macro"])
    ax.set_ylim(0, 1)
    ax.set_ylabel("F1 macro (mismo test)")
    ax.set_title("A · Cambiar n-gramas sin cambiar prueba")
    fig.tight_layout()
    fig.savefig(carpeta_evidencia / "s13_A_f1_ngramas.png", dpi=125)
    plt.show()

    reporte = predicciones_por_config["Unigramas + bigramas"]
    f1_clase = pd.Series({c: reporte[c]["f1-score"] for c in etiquetas_fijas})
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.bar(f1_clase.index, f1_clase.values)
    ax.set_ylim(0, 1)
    ax.set_ylabel("F1 por segmento")
    ax.set_xlabel("Segmento UNSPSC declarado")
    ax.set_title("B · Desempeño desigual entre clases")
    fig.tight_layout()
    fig.savefig(carpeta_evidencia / "s13_B_f1_por_clase.png", dpi=125)
    plt.show()
    print("Mismo test SHA-256:", test_hash)
    print("Evidencia lista:", carpeta_evidencia)


**Cómo se lee:** A compara las F1 macro de dos n-gramas; B muestra qué clases quedan mal representadas (valores cercanos a 0). El CSV conserva exactamente el hash del test y los tamaños train/test.  
**Qué nos dice:** una configuración puede mejorar el promedio, pero afectar de forma distinta a categorías minoritarias.  
**Qué NO permite concluir todavía:** que ajustar parámetros reiteradamente sobre el test final sea una evaluación imparcial; para seleccionar hiperparámetros se necesita validación separada del test.  
**Error común:** interpretar la clase con F1 más alta como «más legal» o tomar dos modelos con test distintos como comparables.

**Solo si activaste la ampliación:** puedes conservar los gráficos, la tabla CSV y su código. **Producto principal de S13:** ejecución Dask con workers y tiempos reales, Parquet y decisión cerrada. No se requiere ejecutar inferencias grandes ni redactar reflexiones abiertas.


**Cómo se lee:** F1 macro cercano a 1 sería mejor; en este conjunto pequeño importa compararlo con el control y revisar la F1 de las categorías minoritarias.  
**Qué nos dice:** si el texto aporta señales para aproximar la etiqueta administrativa.  
**Qué NO permite concluir todavía:** que el sistema generalice a SECOP completo ni que esté validado para hacer revisiones jurídicas.  
**Error común:** reportar sólo exactitud general cuando el segmento 80 domina.

**Reto:** ajusta sólo `ngram_range=(1,1)` y compara la calidad sobre el **mismo test**. No ajustes los hiperparámetros observando repetidamente el test final.


### Práctica de transferencia · Dask y representaciones explicables

Abre el Codespace S13 desde `Code → Codespaces → New with options`; ejecuta `nproc`, `free -h` y `python Laboratorios/S13_Codespaces/dask_cuando_si.py --workers 2`. El resultado guarda un JSON con CPU disponibles, workers realmente creados y tres tiempos.

Para TF-IDF, abre Streamlit en el puerto 8503 **Private**. Modifica una palabra y observa cómo cambian DF, IDF, TF(A), TF×IDF(A) y L2(A). Los dos gráficos de clasificación pertenecen a la ampliación opcional; no vuelvas a correr 50.000 procesos.

**Rúbrica observable de práctica (10 puntos, retroalimentación formativa):**

| Criterio | 2 puntos | 1 punto | 0 puntos |
|---|---|---|---|
| Recursos | Identifica CPU habilitadas, RAM y comando | Identifica solo un recurso | Sin evidencia |
| Workers | Interpreta procesos, hilos y direcciones del scheduler | Solo número de workers | Confunde workers con nodos |
| Igualdad | Muestra `resultados_identicos=True` | Resultado sin comprobación | Resultados distintos |
| Tiempo | Compara secuencial, Dask caliente y total | Omite el arranque | Compara fases diferentes |
| TF-IDF | Explica DF, IDF y L2 con una fila real | Confunde un concepto | No interpreta la tabla |

**Nota:** es una pauta formativa de autocomprobación, no una nueva nota ni un cambio en el PDA. Las validaciones grandes, la depuración de la fuente y la calificación formal las realiza el docente por separado.

**Producto a S14:** artefacto Parquet y un mismo caso que podremos transformar con PySpark.

**Cómo ver la tarea en el panel de Dask:** desde terminal, ejecuta `python Laboratorios/S13_Codespaces/dask_cuando_si.py --workers 2 --tablero --mantener-tablero-segundos 30` y abre **PORTS → 8787 → Private** mientras se mantiene abierto. El período de observación NO se suma al tiempo de cómputo.


In [ ]:
# Reto de cierre · respuesta estructurada y verificable (sin texto libre)
# Modifica SOLO los dos números según TUS mediciones. Ninguno está preseleccionado.
# ARQUITECTURA: 1 = pandas/TF-IDF local; 2 = Dask para lotes independientes;
# 3 = medir más antes de distribuir.
# LIMITE: 1 = no contamos arranque del clúster; 2 = muestra no representativa;
# 3 = etiquetas UNSPSC administrativas y desequilibradas;
# 4 = los tiempos observados dependen del hardware.
ARQUITECTURA = None
LIMITE = None

OPCIONES_ARQUITECTURA = {
    1: "Continuar con TF-IDF local",
    2: "Distribuir lotes independientes con Dask (workers medidos)",
    3: "No decidir aún: medir costo extremo a extremo",
}
OPCIONES_LIMITE = {
    1: "Una comparación omite el arranque del clúster",
    2: "La muestra evaluada no representa todos los procesos SECOP",
    3: "UNSPSC es una etiqueta administrativa y las clases son desiguales",
    4: "La comparación se hizo en un hardware concreto",
}
print("Elige UNA arquitectura y UN límite de los siguientes menús:")
print("Arquitectura:", OPCIONES_ARQUITECTURA)
print("Límite:", OPCIONES_LIMITE)
if ARQUITECTURA is None or LIMITE is None:
    print("Práctica pendiente: cambia ambos None por los índices seleccionados y vuelve a ejecutar.")
else:
    if type(ARQUITECTURA) is not int or ARQUITECTURA not in OPCIONES_ARQUITECTURA:
        raise ValueError("ARQUITECTURA debe ser exactamente 1, 2 o 3")
    if type(LIMITE) is not int or LIMITE not in OPCIONES_LIMITE:
        raise ValueError("LIMITE debe ser exactamente 1, 2, 3 o 4")
    import json
    from pathlib import Path
    evidencia = {
        "sesion": 13,
        "arquitectura_codigo": ARQUITECTURA,
        "arquitectura": OPCIONES_ARQUITECTURA[ARQUITECTURA],
        "limite_codigo": LIMITE,
        "limite": OPCIONES_LIMITE[LIMITE],
        "fuente": "mediciones propias del cuaderno / resultados históricos diferenciados"
    }
    salida_decision = RAIZ / "Laboratorios/S13_Codespaces/salidas/decision_s13.json"
    salida_decision.parent.mkdir(parents=True, exist_ok=True)
    salida_decision.write_text(json.dumps(evidencia,ensure_ascii=False,indent=2), encoding="utf-8")
    print("Evidencia exportada a:", salida_decision)
    print("Comprueba que la arquitectura seleccionada corresponda a tus tiempos, no a un resultado supuesto.")


## Cierre · Puente a la sesión 14

El resultado de la sesión 13 es `secop_s13.parquet`, un CSV fuente identificado, evidencias de workers y coordinación Dask. La clasificación UNSPSC, si la activaste, es opcional. **En S14 usaremos PySpark para leer el mismo conjunto y comparar un tercer motor de procesamiento**, no volver a empezar otro caso.

**Errores comunes que evitamos:** usar Dask para 200 filas sin justificarlo; contar modalidades como irregularidades; promediar promedios; llamar «clúster físico» a dos workers locales; ajustar TF-IDF antes del corte; confundir inferencia y entrenamiento; comparar tiempos sin mismo trabajo.

**Fuentes oficiales:** [Dask](https://docs.dask.org/en/stable/) · [Dask Distributed](https://distributed.dask.org/) · [scikit-learn](https://scikit-learn.org/stable/) · [Codespaces](https://docs.github.com/es/codespaces).
